# Módulo 10. Sistema de Alertas Tempranas

Como aplicación práctica del modelo desarrollado, se implementó un sistema de alertas tempranas capaz de clasificar automáticamente a los municipios según el nivel de riesgo de deserción escolar estimado por el modelo de aprendizaje automático.

El objetivo de este sistema es facilitar la identificación de territorios que requieren una mayor atención por parte de las autoridades educativas, permitiendo priorizar estrategias de intervención y seguimiento.

Para la clasificación se definieron tres niveles de riesgo de acuerdo con la tasa de deserción predicha por el modelo:

- 🟢 **Bajo riesgo:** menor al 4%.
- 🟡 **Riesgo medio:** entre 4% y 8%.
- 🔴 **Alto riesgo:** superior al 8%.

In [9]:
import pandas as pd
import numpy as np
import joblib

from pathlib import Path

In [10]:
PROJECT_ROOT = Path.cwd().parent

df = pd.read_csv(
    PROJECT_ROOT /
    "data" /
    "processed" /
    "dataset_modelado.csv"
)

print(df.shape)
df.head()

(15707, 85)


,AÑO,CÓDIGO_MUNICIPIO,MUNICIPIO,CÓDIGO_DEPARTAMENTO,DEPARTAMENTO,CÓDIGO_ETC,ETC,POBLACIÓN_5_16,TASA_MATRICULACIÓN_5_16,COBERTURA_NETA,...,REPITENCIA_MEDIA_ETC,BENEFICIARIOS_PAE,BRECHA_COBERTURA,BRECHA_APROBACION,INDICE_EFICIENCIA,PRESION_SISTEMA,DIGITALIZADO,TAM_GRUPO_NORMALIZADO,PESO_MUNICIPIO_ETC,PANDEMIA
0,2024,5004,Abriaquí,5,Antioquia,3758.0,Antioquia (ETC),499.0,56.11,56.11,...,3.35,NaN,5.81,99.36,11.418099,8.893245,0,NaN,0.001007,0
1,2024,15204,Cómbita,15,Boyacá,3769.0,Boyacá (ETC),1862.0,95.33,95.33,...,3.21,NaN,96.18,91.52,6.821739,19.532151,0,NaN,0.011952,0
2,2024,99773,Cumaribo,99,Vichada,3832.0,Vichada (ETC),25239.0,50.70,50.70,...,4.08,NaN,7.04,64.06,1.940284,497.810651,0,NaN,0.750245,0
3,2024,99624,Santa Rosalía,99,Vichada,3832.0,Vichada (ETC),1157.0,81.42,81.42,...,4.08,NaN,9.16,78.44,3.176015,14.210268,0,NaN,0.034393,0
4,2024,99524,La Primavera,99,Vichada,3832.0,Vichada (ETC),2645.0,90.96,90.96,...,4.08,NaN,8.17,77.18,3.169123,29.078716,0,NaN,0.078624,0


## Resultados del sistema de alertas

La siguiente tabla presenta una muestra de las predicciones generadas por el modelo para diferentes municipios del país, junto con el nivel de riesgo asignado automáticamente.

Esta clasificación constituye una herramienta de apoyo para la toma de decisiones, permitiendo identificar de manera rápida aquellos municipios que podrían requerir intervenciones preventivas.

In [11]:
modelo = joblib.load(
    PROJECT_ROOT /
    "models" /
    "random_forest_desercion.pkl"
)

print(type(modelo))

<class 'sklearn.ensemble._forest.RandomForestRegressor'>


In [14]:
list(modelo.feature_names_in_)

['AÑO',
 'CÓDIGO_ETC',
 'POBLACIÓN_5_16',
 'TASA_MATRICULACIÓN_5_16',
 'COBERTURA_NETA',
 'COBERTURA_NETA_TRANSICIÓN',
 'COBERTURA_NETA_PRIMARIA',
 'COBERTURA_NETA_SECUNDARIA',
 'COBERTURA_NETA_MEDIA',
 'COBERTURA_BRUTA',
 'COBERTURA_BRUTA_TRANSICIÓN',
 'COBERTURA_BRUTA_PRIMARIA',
 'COBERTURA_BRUTA_SECUNDARIA',
 'COBERTURA_BRUTA_MEDIA',
 'TAMAÑO_PROMEDIO_DE_GRUPO',
 'SEDES_CONECTADAS_A_INTERNET',
 'APROBACIÓN',
 'APROBACIÓN_TRANSICIÓN',
 'APROBACIÓN_PRIMARIA',
 'APROBACIÓN_SECUNDARIA',
 'APROBACIÓN_MEDIA',
 'REPROBACIÓN',
 'REPROBACIÓN_TRANSICIÓN',
 'REPROBACIÓN_PRIMARIA',
 'REPROBACIÓN_SECUNDARIA',
 'REPROBACIÓN_MEDIA',
 'REPITENCIA',
 'REPITENCIA_TRANSICIÓN',
 'REPITENCIA_PRIMARIA',
 'REPITENCIA_SECUNDARIA',
 'REPITENCIA_MEDIA',
 'POBLACIÓN_5_16_ETC',
 'TASA_MATRICULACIÓN_5_16_ETC',
 'COBERTURA_NETA_ETC',
 'COBERTURA_NETA_TRANSICIÓN_ETC',
 'COBERTURA_NETA_PRIMARIA_ETC',
 'COBERTURA_NETA_SECUNDARIA_ETC',
 'COBERTURA_NETA_MEDIA_ETC',
 'COBERTURA_BRUTA_ETC',
 'COBERTURA_BRUTA_TRANSICIÓN

In [15]:
# ============================================
# Preparar datos exactamente como los espera el modelo
# ============================================

# Tomar únicamente las variables con las que fue entrenado
X = df.loc[:, modelo.feature_names_in_].copy()

# Imputar valores faltantes igual que en el entrenamiento
X = X.fillna(X.median(numeric_only=True))

print(X.shape)
X.head()

(15707, 69)


,AÑO,CÓDIGO_ETC,POBLACIÓN_5_16,TASA_MATRICULACIÓN_5_16,COBERTURA_NETA,COBERTURA_NETA_TRANSICIÓN,COBERTURA_NETA_PRIMARIA,COBERTURA_NETA_SECUNDARIA,COBERTURA_NETA_MEDIA,COBERTURA_BRUTA,...,REPITENCIA_MEDIA_ETC,BENEFICIARIOS_PAE,BRECHA_COBERTURA,BRECHA_APROBACION,INDICE_EFICIENCIA,PRESION_SISTEMA,DIGITALIZADO,TAM_GRUPO_NORMALIZADO,PESO_MUNICIPIO_ETC,PANDEMIA
0,2024,3758.0,499.0,56.11,56.11,39.53,59.13,51.52,26.51,61.92,...,3.35,2394.0,5.81,99.36,11.418099,8.893245,0,1.000325,0.001007,0
1,2024,3769.0,1862.0,95.33,95.33,53.64,89.15,98.88,62.10,191.51,...,3.21,2394.0,96.18,91.52,6.821739,19.532151,0,1.000325,0.011952,0
2,2024,3832.0,25239.0,50.70,50.70,40.14,57.71,16.95,3.85,57.74,...,4.08,2394.0,7.04,64.06,1.940284,497.810651,0,1.000325,0.750245,0
3,2024,3832.0,1157.0,81.42,81.42,70.67,82.64,54.90,22.45,90.58,...,4.08,2394.0,9.16,78.44,3.176015,14.210268,0,1.000325,0.034393,0
4,2024,3832.0,2645.0,90.96,90.96,64.13,96.46,59.01,31.33,99.13,...,4.08,2394.0,8.17,77.18,3.169123,29.078716,0,1.000325,0.078624,0


In [16]:
predicciones = modelo.predict(X)

print(predicciones[:10])

[0.1589   3.222    5.7153   6.2962   5.2048   8.4671   9.4108   0.846417
 0.       7.6402  ]


In [17]:
# Crear tabla de alertas
alertas = df[["AÑO", "MUNICIPIO", "DEPARTAMENTO"]].copy()

alertas["Predicción"] = predicciones

# Clasificación del riesgo
def clasificar_riesgo(valor):
    if valor < 4:
        return "🟢 Bajo"
    elif valor < 8:
        return "🟡 Medio"
    else:
        return "🔴 Alto"

alertas["Nivel de riesgo"] = alertas["Predicción"].apply(clasificar_riesgo)

alertas.head(10)

,AÑO,MUNICIPIO,DEPARTAMENTO,Predicción,Nivel de riesgo
0,2024,Abriaquí,Antioquia,0.158900,🟢 Bajo
1,2024,Cómbita,Boyacá,3.222000,🟢 Bajo
2,2024,Cumaribo,Vichada,5.715300,🟡 Medio
3,2024,Santa Rosalía,Vichada,6.296200,🟡 Medio
4,2024,La Primavera,Vichada,5.204800,🟡 Medio
5,2024,Puerto Carreño,Vichada,8.467100,🔴 Alto
6,2024,Yavaraté,Vaupés,9.410800,🔴 Alto
7,2024,Papunaua,Vaupés,0.846417,🟢 Bajo
8,2024,Taraira,Vaupés,0.000000,🟢 Bajo
9,2024,Pacoa,Vaupés,7.640200,🟡 Medio


## Distribución de los niveles de riesgo

A continuación se muestra la cantidad de registros clasificados en cada nivel de riesgo. Esta distribución permite obtener una visión general del comportamiento de la deserción escolar predicha a nivel nacional.

In [18]:
alertas["Nivel de riesgo"].value_counts()

Nivel de riesgo
🟢 Bajo     10332
🟡 Medio     4738
🔴 Alto       637
Name: count, dtype: int64

## Municipios con mayor riesgo de deserción escolar

Finalmente, se presentan los municipios con las mayores tasas de deserción predichas por el modelo.

Estos resultados permiten identificar territorios que podrían beneficiarse de programas de intervención prioritaria y representan un ejemplo del potencial del sistema de alertas tempranas desarrollado en este proyecto.

In [19]:
alertas.sort_values(
    by="Predicción",
    ascending=False
).head(20)

,AÑO,MUNICIPIO,DEPARTAMENTO,Predicción,Nivel de riesgo
3380,2021,Morichal,Guainía,21.7217,🔴 Alto
9333,2016,Morichal,Guainía,15.0301,🔴 Alto
2259,2022,La Guadalupe,Guainía,14.7151,🔴 Alto
2236,2023,San Felipe,Guainía,14.4766,🔴 Alto
7837,2018,Pana Pana,Guainía,14.4030,🔴 Alto
3381,2021,Pana Pana,Guainía,14.3579,🔴 Alto
3869,2021,Hobo,Huila,14.3555,🔴 Alto
3408,2021,Puerto Guzmán,Putumayo,14.1754,🔴 Alto
7839,2018,La Guadalupe,Guainía,14.1056,🔴 Alto
14606,2011,La Guadalupe,Guainía,13.9855,🔴 Alto


## Municipios priorizados para intervención (Año 2024)

Con el fin de ilustrar una aplicación práctica del sistema de alertas tempranas, se presentan los municipios con las mayores tasas de deserción escolar predichas para el año 2024.

Este ranking permite priorizar territorios que podrían requerir un seguimiento más cercano por parte de las autoridades educativas, facilitando la asignación de recursos y el diseño de estrategias preventivas orientadas a reducir la deserción escolar.

Los resultados muestran una mayor concentración de municipios de alto riesgo en departamentos como Amazonas, Guainía, Vaupés, Putumayo, Meta y Caquetá, lo que coincide con patrones identificados previamente durante el análisis exploratorio y la validación del modelo.

In [20]:
# Municipios con mayor riesgo en el año más reciente

alertas_2024 = alertas[alertas["AÑO"] == 2024]

alertas_2024.sort_values(
    by="Predicción",
    ascending=False
).head(20)

,AÑO,MUNICIPIO,DEPARTAMENTO,Predicción,Nivel de riesgo
32,2024,La Chorrera,Amazonas,13.7813,🔴 Alto
425,2024,Mapiripán,Meta,10.6495,🔴 Alto
44,2024,Puerto Guzmán,Putumayo,10.4326,🔴 Alto
16,2024,Morichal,Guainía,10.1840,🔴 Alto
6,2024,Yavaraté,Vaupés,9.4108,🔴 Alto
420,2024,Puerto Concordia,Meta,8.8566,🔴 Alto
772,2024,Morelia,Caquetá,8.8311,🔴 Alto
674,2024,San José de Uré,Córdoba,8.6071,🔴 Alto
280,2024,Santuario,Risaralda,8.5080,🔴 Alto
5,2024,Puerto Carreño,Vichada,8.4671,🔴 Alto
